# Conclusiones del Proyecto: ETL Presuntos Suicidios Colombia 2015-2024

**Autores:** Ana María Toro · Ana Sofía Perhueza · Daniel Gómez Sermeño · Daniel Steven Colmenares
----

## 1. Resumen

- **Arquitectura medallón**: Bronze (crudo) → Silver (limpio) → Gold (con métricas)
- **Fuentes**: API INMLCF (26.558 suicidios) + Excel DANE (población departamental y municipal, total y por sexo y edad)
- **Pipeline**: `python main.py` corre todo (extract → transform → load)
- **Tiempo**: ~80 segundos con cache (la mayor parte es leer los Excel municipales del DANE por sexo y edad)
- **Tests**: 24/24 pasan

### Arquitectura Medallón

| Capa | Propósito | Ubicación |
|------|-----------|-----------|
| **Bronze** | Datos crudos inmutables (API + Excel) | `data/bronze/` |
| **Silver** | Tablas limpias por fuente (estandarización sin pérdida) | `data/silver/` |
| **Gold** | Base integrada con métricas/KPIs para análisis | `data/gold/` |

### Flujo del Pipeline

```
main.py (orquestador único)
    ↓
src/extract/extract_data.py  →  data/bronze/
    ↓
src/transform/clean_data.py  →  data/silver/
    ↓
src/transform/gold_data.py   →  data/gold/
    ↓
src/load/load_data.py        →  CSV + SQLite (o PostgreSQL)
```

### Principios Clave

- **Bronze nunca se modifica** — datos crudos son inmutables
- **Configuración centralizada** — todo en `config/config.yaml`
- **Logging estructurado** — `get_logger(__name__)` en consola y `logs/etl_log.txt`
- **Código en español sin tildes** en identificadores
- **Población DANE no se une fila a fila** — se agrega por depto/año para tasas por 100k hab.

## 2. Lo que encontramos en los datos crudos (EDA)

- **26.558 registros**, 32 columnas, años 2015-2024 ✓ (coincide con la cifra oficial)
- **No hay nulos vacíos**: lo que falta viene como "Sin información" (categoría)
- **2 columnas constantes** que no sirven: `circunstancia_del_hecho_detallada` y `manera_de_muerte`
- **Casi no hay duplicados**: solo 1 par de filas iguales (distinto `id`)
- **Muchas variantes de escritura** según el año (meses con mayúscula, Bogotá D.C. vs Bogotá, D.C., Quindio vs Quindío, etc.)
- **Escolaridad** tiene dos clasificaciones mezcladas según el año
- **Códigos DANE** son clave para unir con población (depto 2 dígitos, municipio 5)

## 3. Clasificación de mecanismo (para la paradoja de género)

Según el anexo del documento, clasificamos el mecanismo en 3 grupos:

| Tipo | Casos | Qué incluye |
|------|-------|-------------|
| **Violento** | 22.288 | Ahorcamiento, arma de fuego, cortante, contundente, explosivo, etc. |
| **No violento** | 4.235 | Tóxico, cáustico |
| **Por determinar** | 35 | Lo que no encaja |

→ **Coincide exacto con la Tabla 4 del documento**.

## 4. Agrupación de razones del suicidio

- 52.9% = "Sin información" (mucho subregistro)
- De los que sí tienen razón, **9 categorías concentran casi todo**:
  1. Conflicto de pareja / ex-pareja
  2. Enfermedad física o mental
  3. Enfermedad mental
  4. Desamor
  5. Económicas
  6. Enfermedad física
  7. Abuso de sustancias
  8. Muerte de familiar o amigo
  9. Jurídicas
- El resto (bullying, ciberacoso, orientación sexual, acceso a armas...) son muy pocos casos → se agrupan en "Otras"
- En Gold creamos `razon_agrupada` con: 9 principales + Otras + Sin información

## 5. Población DANE

- Población total departamental: un archivo unificado 2005-2050 (retroproyecciones 2005-2017 + proyecciones post-COVID 2018-2050)
- Para tasas por sexo y edad y para municipios se agregaron las proyecciones DANE departamentales por sexo y edad simple y las municipales (total, y por sexo y edad para el Valle del Cauca)
- Se filtran: solo `area_geografica == "total"` y códigos depto válidos (2 dígitos)
- Población viene como texto con separador de miles → entero
- Esquema final silver: `codigo_dane`, `departamento`, `ano`, `poblacion`
- **Un nombre por código DANE** (arregla inconsistencias de escritura)

## 6. Transformaciones Silver (lo que hicimos)

**`limpiar_suicidios()`:**
1. Unifica esquema de columnas (`a_o_del_hecho` ↔ `ano_del_hecho`)
2. Elimina columnas constantes
3. Códigos DANE con cero a la izquierda
4. `unificar_categorias()` en variables categóricas (excepto geográficas)
5. `unificar_por_codigo()` en nombres geográficos (depto/municipio)
6. Normaliza redacciones específicas:
   - Mecanismo explosivo → "Agente o mecanismo explosivo"
   - Mayor/menor de edad → "Mayor de edad" / "Menor de edad"
7. Depto sin dato (código 999) → código = "Sin informacion"
8. Tipos: `id` y `ano_del_hecho` enteros; años fuera de rango → nulo
9. Validación: avisa incoherencia grupo quinquenal vs mayor/menor

**`limpiar_poblacion_2005_2050()`:**
1. Filtra notas al pie y áreas ≠ total
2. Población texto → entero
3. Esquema: `codigo_dane`, `departamento`, `ano`, `poblacion`
4. Unifica nombre depto por código DANE

## 7. Qué generó la capa Gold

La tabla de casos (`gold_suicidios`) tiene **26.558 filas × 27 columnas**: las variables de silver que se usan en el análisis más estas columnas nuevas (las variables sensibles que no se usan quedan fuera por privacidad):

| Columna nueva | Qué es | Valores |
|---------------|--------|---------|
| `tipo_mecanismo` | Violento / No violento / Por determinar | 22.288 / 4.235 / 35 |
| `es_violento` | Boolean | True / False |
| `razon_agrupada` | 9 categorías + Otras + Sin info | Ver arriba |
| `es_mayor_edad` | Boolean | True: 23.768, False: 2.790 |
| `escolaridad_agrupada` | Escolaridad homologada (las dos clasificaciones de la fuente) | 8 niveles + Sin información |
| `region` | Región natural del departamento | Andina, Caribe, Pacífica, Orinoquía, Amazonía |

**La tasa ya no va en la tabla de casos:** se calcula en `gold_tasa_mortalidad` (casos / población × 100.000) por territorio, periodo, sexo y grupo de edad. Pegar la tasa a cada caso hacía que, al promediarla, los años con más casos pesaran más (ver la corrección de Vaupés en `informe.ipynb`).

## 8. Validaciones: todo cuadra

-  **Tabla 4 del documento**: 22.288 violentos, 4.235 no violentos, 35 por determinar → **EXACTO**
-  **Paradoja de género**: Hombres 86.7% mecanismos violentos vs Mujeres 73.2% (sin los 35 'Por determinar', igual que la Tabla 4)
-  **Tasas por depto/año**: las 330 combinaciones departamento-año tienen población (los 4 casos sin departamento se excluyen de las tasas departamentales)
-  **Chi-cuadrado viable**: frecuencias esperadas ≥ 5 en todas las celdas (mínimo 12,6); razón × grupo etario da χ² = 2.924,4 con 27 gl, igual que la Tabla 9
-  **Tendencia temporal**: Casos suben 45.7% (2015→2024), tasa sube 28.2% (4,47 → 5,73)
-  **Edad**: Pico en 20-29 años (26.1% del total)
-  **Mayor de edad**: 89.5% de los casos

## 9. Datos interesantes

**Top 5 departamentos por tasa media anual (2015-2024):**
1. Vaupés: 13.01
2. Arauca: 10.34
3. Quindío: 8.17
4. Caldas: 7.89
5. Risaralda: 7.81

(Media de las 10 tasas anuales, incluidos los años con 0 casos.)

**Distribución por sexo:**
- Hombres: 21.349 (80.4%)
- Mujeres: 5.209 (19.6%)

**Mecanismo por sexo (paradoja de género):**
- Hombres: 18.483 violentos, 2.843 no violentos
- Mujeres: 3.805 violentos, 1.392 no violentos

**Casos por año:** 2.068 (2015) → 3.014 (2024), subida sostenida

## 10. Limitaciones que vimos

1. **Mucho subregistro en razón** (52.9% sin info) → limita análisis causal
2. **No hay edad exacta ni fecha exacta** → solo grupos quinquenales y mes/día semana
3. **Escolaridad** tiene dos clasificaciones según año → se homologó en `escolaridad_agrupada`, pero la categoría preescolar es sospechosamente alta
4. **Población DANE son proyecciones**, no censos; corte 2018 entre retro/proyección
5. **Código 999** = depto sin informar (4 casos) → se excluyen de tasas
6. **Municipio**: casi todos tienen pocos casos → tasas solo para 6 municipios foco del Valle; el resto, conteos agregados